In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import cftime
import os

In [2]:
# ============== LOAD LLC ==============
llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
llc_patch_full = xr.open_dataset(llc_path, consolidated=True)

# ============== LOAD EMULATORS ==============
emulator_configs = [
    {
    #     'name': 'strides=1,ckpt_4',
    #     'key': 'emulator_1',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=1,ckpt_4/predictions_4d.zarr',
    #     'desc': 'strides=1'
    # },
    # {
    #     'name': 'strides=1,ckpt_8',
    #     'key': 'emulator_2',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-18-eval:Samudra_LLC:strides=1,ckpt_8/predictions_4d.zarr',
    #     'desc': 'strides=1'
    # },
        # {
        'name': 'mae+mse+grad_long_curriculum(speed_test_C)_ckpt22',
        'key': 'emulator_3',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-22-eval:Samudra_LLC:mae+mse+grad_long_curriculum(speed_test_C)_ckpt22-14276428/predictions_4d.zarr',
        'desc': 'mae+mse+grad_long_curriculum(speed_test_C)_ckpt22'
    },
    {
        'name': 'mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22',
        'key': 'emulator_4',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-05-22-eval:mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22-14276515/predictions_4d.zarr',
        'desc': 'mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22'
    },
]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    for gv in grid_vars:
        patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded mae+mse+grad_long_curriculum(speed_test_C)_ckpt22: mae+mse+grad_long_curriculum(speed_test_C)_ckpt22
Loaded mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22: mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22
LLC subset to 16 common times

=== Setup complete: LLC + 2 emulators ===
  mae+mse+grad_long_curriculum(speed_test_C)_ckpt22 (emulator_3)
  mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22 (emulator_4)


In [3]:
selected_time_range = [0, 16]   # inclusive indices
stepping = 1                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 0:16
Stepping = 1
Final synchronized length = 16
LLC now has 16 times
mae+mse+grad_long_curriculum(speed_test_C)_ckpt22 (emulator_3) now has 16 times
mae+mse+grad_long_curriculum(GRAD_H+GRAD_Z-TEST)_ckpt22 (emulator_4) now has 16 times


In [4]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

# surface field and field difference

In [5]:
# ============== SET VARIABLES HERE ==============
# prog_vars = ['Theta', 'Salt', 'U', 'V']
# colormaps = {'Theta': 'Spectral_r', 'Salt': 'viridis', 'U': 'bwr', 'V': 'bwr'}

prog_vars = ['Theta']
colormaps = {'Theta': 'Spectral_r'}
# ================================================
for var in prog_vars:
    print(f"Generating plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols_fields = 1 + n_emulators  # LLC + emulators
    ncols_diff = n_emulators         # emulators only
    
    cmap = colormaps[var]
    
    # ==================== PLOT 1: Surface fields ====================
    fig, axes = plt.subplots(nrows, ncols_fields, figsize=(3.6*ncols_fields, 3*nrows), dpi=200)
    
    if nrows == 1:
        axes = axes.reshape(1, -1)
    if ncols_fields == 1:
        axes = axes.reshape(-1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        # Collect all surface fields
        fields = [llc_patch.isel(time=t, k=0)[var]]
        for emu_name, emu_key in emulator_info:
            fields.append(emulator_patches[emu_key].isel(time=t, k=0)[var])
        
        vmin = np.min([f.values.min() for f in fields])
        vmax = np.max([f.values.max() for f in fields])
        
        labels = ['LLC'] + [name for name, _ in emulator_info]
        
        for col, (field, label) in enumerate(zip(fields, labels)):
            ax = axes[row, col]
            cf = ax.contourf(field.coords.get('i', np.arange(field.shape[-1])),
                             field.coords.get('j', np.arange(field.shape[-2])), field,
                             cmap=cmap, vmin=vmin, vmax=vmax, levels=30)
            ax.set_title(f'{label} {var} {time_str}', fontsize=8)
            plt.colorbar(cf, ax=ax)
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_fields.png')
    plt.close()
    
    # ==================== PLOT 2: Difference fields ====================
    fig, axes = plt.subplots(nrows, ncols_diff, figsize=(4*ncols_diff, 3*nrows), dpi=200)
    
    if nrows == 1 and ncols_diff > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols_diff == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols_diff == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_vis = llc_patch.isel(time=t, k=0)[var]
        
        diffs = []
        for emu_name, emu_key in emulator_info:
            emu_vis = emulator_patches[emu_key].isel(time=t, k=0)[var]
            diffs.append(llc_vis.values - emu_vis.values)
        
        abs_max = np.max([np.abs(d).max() for d in diffs])
        vmin_d, vmax_d = -abs_max, abs_max
        
        row_axes = [axes[row, col] for col in range(ncols_diff)]
        
        for col, ((emu_name, _), diff) in enumerate(zip(emulator_info, diffs)):
            ax = row_axes[col]
            cf = ax.contourf(llc_vis.coords.get('i', np.arange(llc_vis.shape[-1])),
                             llc_vis.coords.get('j', np.arange(llc_vis.shape[-2])), diff,
                             cmap="bwr", vmin=vmin_d, vmax=vmax_d, levels=30)
            short_name = emu_name.replace('Emulator ', 'Em')
            ax.set_title(f'LLC - {short_name} {var} {time_str}', fontsize=8)
        
        fig.colorbar(cf, ax=row_axes, orientation='vertical',
                     fraction=0.046, pad=0.04)
    
    plt.savefig(f'figs/prognostic_var_comparison/{var}/surface_{var}_differences.png')
    plt.close()
    
    print(f"✓ Saved plots for {var}")

Generating plots for Theta...
✓ Saved plots for Theta


# Gradients

In [6]:
grad_vars = ['Theta', 'Salt', 'U', 'V']

for var in grad_vars:
    grad_name = f'grad_{var}'
    print(f"Computing {grad_name}...")
    
    for patch_name, patch in all_patches.items():
        data = patch[var].values  # (time, k, j, i)
        
        dx = np.sqrt(patch['rA'].values)  # (j, i) in meters
        dy = dx.copy()
        
        d_di = (np.roll(data, -1, axis=3) - np.roll(data, 1, axis=3)) / (2 * dx[np.newaxis, np.newaxis, :, :])
        d_dj = (np.roll(data, -1, axis=2) - np.roll(data, 1, axis=2)) / (2 * dy[np.newaxis, np.newaxis, :, :])
        
        grad_mag = np.sqrt(d_di**2 + d_dj**2)
        
        patch[grad_name] = (('time', 'k', 'j', 'i'), grad_mag)
        print(f"  ✓ {patch_name} {grad_name}: {grad_mag.shape}")

print("Done computing gradients!")

Computing grad_Theta...
  ✓ llc grad_Theta: (16, 51, 720, 720)
  ✓ emulator_3 grad_Theta: (16, 51, 720, 720)
  ✓ emulator_4 grad_Theta: (16, 51, 720, 720)
Computing grad_Salt...
  ✓ llc grad_Salt: (16, 51, 720, 720)
  ✓ emulator_3 grad_Salt: (16, 51, 720, 720)
  ✓ emulator_4 grad_Salt: (16, 51, 720, 720)
Computing grad_U...
  ✓ llc grad_U: (16, 51, 720, 720)
  ✓ emulator_3 grad_U: (16, 51, 720, 720)
  ✓ emulator_4 grad_U: (16, 51, 720, 720)
Computing grad_V...
  ✓ llc grad_V: (16, 51, 720, 720)
  ✓ emulator_3 grad_V: (16, 51, 720, 720)
  ✓ emulator_4 grad_V: (16, 51, 720, 720)
Done computing gradients!


In [7]:
gradient_masks = {}

for patch_name, patch in all_patches.items():
    gradient_masks[patch_name] = {}
    
    for var in grad_vars:
        grad_name = f'grad_{var}'
        grad_data = patch[grad_name].values  # (time, k, j, i)
        
        n_times, n_depths = grad_data.shape[0], grad_data.shape[1]
        mask = np.zeros_like(grad_data, dtype=bool)
        
        for t in range(n_times):
            for k in range(n_depths):
                field = grad_data[t, k]
                threshold = np.nanpercentile(field, 97.5)
                mask[t, k] = field >= threshold
        
        gradient_masks[patch_name][var] = mask
        print(f"✓ {patch_name} {var}: {mask.sum()} high-gradient pixels ({mask.sum() / mask.size * 100:.1f}%)")

print("Done creating gradient masks!")

✓ llc Theta: 10574832 high-gradient pixels (2.5%)
✓ llc Salt: 10574838 high-gradient pixels (2.5%)
✓ llc U: 10574642 high-gradient pixels (2.5%)
✓ llc V: 10574745 high-gradient pixels (2.5%)
✓ emulator_3 Theta: 10575411 high-gradient pixels (2.5%)
✓ emulator_3 Salt: 10575426 high-gradient pixels (2.5%)
✓ emulator_3 U: 10575413 high-gradient pixels (2.5%)
✓ emulator_3 V: 10575409 high-gradient pixels (2.5%)
✓ emulator_4 Theta: 10575415 high-gradient pixels (2.5%)
✓ emulator_4 Salt: 10575414 high-gradient pixels (2.5%)
✓ emulator_4 U: 10575420 high-gradient pixels (2.5%)
✓ emulator_4 V: 10575414 high-gradient pixels (2.5%)
Done creating gradient masks!


In [8]:
for var in grad_vars:
    print(f"Generating gradient drift figure for {var}...")
    
    os.makedirs(f'figs/gradients/{var}', exist_ok=True)
    
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_mask_surface = gradient_masks['llc'][var][t, 0]  # (j, i)
        
        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            
            emu_field = all_patches[emu_key].isel(time=t, k=0)[var].values
            emu_mask_surface = gradient_masks[emu_key][var][t, 0]
            
            ax.imshow(emu_field, cmap='Greys', aspect='auto', origin='lower')
            
            overlap_mask = llc_mask_surface & emu_mask_surface
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            emu_only_mask = emu_mask_surface & ~llc_mask_surface
            
            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)
            
            ax.scatter(llc_i, llc_j, c='red', s=1, alpha=0.5, label='LLC top 2.5%', rasterized=True)
            ax.scatter(emu_i, emu_j, c='green', s=1, alpha=0.5, label='Emu top 2.5%', rasterized=True)
            ax.scatter(ovl_i, ovl_j, c='yellow', s=1, alpha=0.7, label='Overlap', rasterized=True)
            
            n_overlap = np.sum(overlap_mask)
            
            ax.set_title(f'{emu_name} {var} {time_str} overlap={n_overlap}', fontsize=8)
            ax.tick_params(labelsize=6)
            
            if row == 0 and col == 0:
                ax.legend(fontsize=5, loc='upper right', markerscale=5)
    
    plt.tight_layout()
    plt.savefig(f'figs/gradients/{var}/surface_gradient_drift.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved gradient drift figure for {var}")

print("Done with gradient drift figures!")

Generating gradient drift figure for Theta...
✓ Saved gradient drift figure for Theta
Generating gradient drift figure for Salt...
✓ Saved gradient drift figure for Salt
Generating gradient drift figure for U...
✓ Saved gradient drift figure for U
Generating gradient drift figure for V...
✓ Saved gradient drift figure for V
Done with gradient drift figures!


# Error vs depth plots

In [9]:
depth_vars = ['Theta', 'Salt', 'U', 'V']
ref_lines = {
    'Theta': [0.5, 1.0],
    'Salt': [0.06, 0.12],
    'U': [0.075, 0.15],
    'V': [0.075, 0.15]
}

for var in depth_vars:
    print(f"Generating augmented depth error plots for {var}...")
    
    os.makedirs(f'figs/prognostic_var_comparison/{var}', exist_ok=True)
    
    n_depths = llc_patch.sizes['k']
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = list(range(n_times))
    
    nrows = len(time_indices)
    ncols = n_emulators
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    
    if nrows == 1 and ncols > 1:
        axes = axes.reshape(1, -1)
    elif nrows > 1 and ncols == 1:
        axes = axes.reshape(-1, 1)
    elif nrows == 1 and ncols == 1:
        axes = axes.reshape(1, 1)
    
    depths = np.arange(n_depths)
    
    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        
        llc_data = llc_patch.isel(time=t)[var].values  # (k, j, i)
        llc_grad_mask = gradient_masks['llc'][var][t]   # (k, j, i)
        
        # First pass: compute all errors for shared xlim
        row_mean_errors = []
        row_median_errors = []
        row_hg_mean_errors = []
        
        for emu_name, emu_key in emulator_info:
            emu_data = emulator_patches[emu_key].isel(time=t)[var].values
            diff = np.abs(llc_data - emu_data)
            
            diff_flat = diff.reshape(n_depths, -1)
            mean_errors = np.nanmean(diff_flat, axis=1)
            median_errors = np.nanmedian(diff_flat, axis=1)
            
            hg_mean_errors = np.zeros(n_depths)
            for k in range(n_depths):
                hg_pixels = diff[k][llc_grad_mask[k]]
                if len(hg_pixels) > 0:
                    hg_mean_errors[k] = np.nanmean(hg_pixels)
                else:
                    hg_mean_errors[k] = np.nan
            
            row_mean_errors.append(mean_errors)
            row_median_errors.append(median_errors)
            row_hg_mean_errors.append(hg_mean_errors)
        
        all_errors = np.concatenate(row_mean_errors + row_median_errors + row_hg_mean_errors)
        xmin = 0
        xmax = np.nanmax(all_errors) * 1.05
        
        # Second pass: plot
        for col, (emu_name, _) in enumerate(emulator_info):
            ax = axes[row, col]
            
            ax.scatter(row_mean_errors[col], depths, color='blue', s=30, alpha=0.7, zorder=3)
            ax.plot(row_mean_errors[col], depths, color='blue', alpha=0.4, linewidth=1.5, label='Mean')
            
            ax.scatter(row_median_errors[col], depths, color='red', s=30, alpha=0.7, zorder=3)
            ax.plot(row_median_errors[col], depths, color='red', alpha=0.4, linewidth=1.5, label='Median')
            
            ax.scatter(row_hg_mean_errors[col], depths, color='green', s=30, alpha=0.7, zorder=3)
            ax.plot(row_hg_mean_errors[col], depths, color='green', alpha=0.4, linewidth=1.5, label='HG Mean')
            
            for ref_val in ref_lines[var]:
                ax.axvline(x=ref_val, color='black', linestyle='--', linewidth=1.5, alpha=0.5, zorder=2)
            
            ax.set_title(f'{emu_name} {var} {time_str}', fontsize=8)
            ax.set_xlabel('Abs Error', fontsize=7)
            ax.set_ylabel('Depth (k)', fontsize=7)
            ax.set_ylim(n_depths - 1, 0)
            ax.set_xlim(xmin, xmax)
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            
            if col == 0:
                ax.legend(fontsize=6, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(f'figs/prognostic_var_comparison/{var}/depth_error_by_time.png', dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved augmented depth error plots for {var}")

print("Done!")

Generating augmented depth error plots for Theta...
✓ Saved augmented depth error plots for Theta
Generating augmented depth error plots for Salt...
✓ Saved augmented depth error plots for Salt
Generating augmented depth error plots for U...
✓ Saved augmented depth error plots for U
Generating augmented depth error plots for V...
✓ Saved augmented depth error plots for V
Done!
